In [1]:
import os
from datasets import load_dataset, Audio
from huggingface_hub import snapshot_download


def get_audio_paths(example):
    for column in example.keys():
        if "Audio" in column and isinstance(example[column], str):
            example[column] = os.path.join(download_dir, example[column])
    return example


# Download
repo_id = "ai4bharat/MANGO"
download_dir = snapshot_download(repo_id=repo_id, repo_type="dataset")
dataset = load_dataset(download_dir, split='Hindi__MUSHRA')
dataset = dataset.map(get_audio_paths)

# Cast audio columns
for column in dataset.column_names:
    if 'Audio' in column:
        dataset = dataset.cast_column(column, Audio())

# Explore
print(dataset)
'''
Dataset({
    features: ['Rater_ID', 'FS2_Score', 'VITS_Score', 'ST2_Score', 'ANC_Score',
               'REF_Score', 'FS2_Audio', 'VITS_Audio', 'ST2_Audio', 'ANC_Audio',
               'REF_Audio'],
    num_rows: 11300
})
'''

# # Print first instance
print(dataset[0])
'''
{'Rater_ID': 389, 'FS2_Score': 16, 'VITS_Score': 76, 'ST2_Score': 28,
 'ANC_Score': 40, 'REF_Score': 100, 'FS2_Audio': {'path': ...
'''

# # Available Splits
dataset = load_dataset(download_dir, split=None)
print("Splits:", dataset.keys())
'''
Splits: dict_keys(['Tamil__MUSHRA_DG_NMR', 'Hindi__MUSHRA_DG',
    'Hindi__MUSHRA_NMR', 'Hindi__MUSHRA_DG_NMR', 'Tamil__MUSHRA_NMR',
    'Tamil__MUSHRA_DG', 'Tamil__MUSHRA', 'Hindi__MUSHRA',
    'English__MUSHRA', 'English_MUSHRA_DG_NMR'])
'''


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching ... files: 0it [00:00, ?it/s]

Generating Tamil__MUSHRA_DG_NMR split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating Hindi__MUSHRA_DG split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating Hindi__MUSHRA_NMR split:   0%|          | 0/10200 [00:00<?, ? examples/s]

...

Generating English__MUSHRA split:   0%|          | 0/900 [00:00<?, ? examples/s]

Generating English__MUSHRA_DG_NMR split:   0%|          | 0/930 [00:00<?, ? examples/s]

Map:   0%|          | 0/11300 [00:00<?, ? examples/s]

Dataset({
    features: ['Rater_ID', 'FS2_Score', 'VITS_Score', 'ST2_Score', 'ANC_Score', 'REF_Score', 'FS2_Audio', 'VITS_Audio', 'ST2_Audio', 'ANC_Audio', 'REF_Audio'],
    num_rows: 11300
})
{'Rater_ID': 389, 'FS2_Score': 16, 'VITS_Score': 76, 'ST2_Score': 28, 'ANC_Score': 40, 'REF_Score': 100, 'FS2_Audio': <datasets.features._torchcodec.AudioDecoder object at 0x7adba7a7fe00>, 'VITS_Audio': <datasets.features._torchcodec.AudioDecoder object at 0x7adba46af6e0>, 'ST2_Audio': <datasets.features._torchcodec.AudioDecoder object at 0x7adba7bb5220>, 'ANC_Audio': <datasets.features._torchcodec.AudioDecoder object at 0x7adba7bbe930>, 'REF_Audio': <datasets.features._torchcodec.AudioDecoder object at 0x7adba47a32c0>}
Splits: dict_keys(['Tamil__MUSHRA_DG_NMR', 'Hindi__MUSHRA_DG', 'Hindi__MUSHRA_NMR', 'Hindi__MUSHRA_DG_NMR', 'Tamil__MUSHRA_NMR', 'Tamil__MUSHRA_DG', 'Tamil__MUSHRA', 'Hindi__MUSHRA', 'English__MUSHRA', 'English__MUSHRA_DG_NMR'])


"\nSplits: dict_keys(['Tamil__MUSHRA_DG_NMR', 'Hindi__MUSHRA_DG', \n    'Hindi__MUSHRA_NMR', 'Hindi__MUSHRA_DG_NMR', 'Tamil__MUSHRA_NMR', \n    'Tamil__MUSHRA_DG', 'Tamil__MUSHRA', 'Hindi__MUSHRA', \n    'English__MUSHRA', 'English_MUSHRA_DG_NMR'])\n"

In [2]:
from transformers import VitsModel, AutoTokenizer
import torch
import scipy.io.wavfile

# 1. Load the model and tokenizer (AI4Bharat / Facebook Hindi VITS)
model_name = "facebook/mms-tts-hin" # High-quality Hindi TTS
model = VitsModel.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. Prepare your text
text = "नमस्ते, आप कैसे हैं? यह एक कृत्रिम आवाज़ है।"
inputs = tokenizer(text, return_tensors="pt")

# 3. Generate the waveform
with torch.no_grad():
    output = model(**inputs).waveform

# 4. Save to a file
scipy.io.wavfile.write("generated_hindi.wav", rate=model.config.sampling_rate, data=output[0].numpy())

print("Speech generation complete! Check 'generated_hindi.wav'.")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/145M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/907 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

Speech generation complete! Check 'generated_hindi.wav'.


In [3]:
from IPython.display import Audio
Audio('generated_hindi.wav')